### Word2Vec Implementation from Scratch

# Introduction
 Word2Vec is a popular technique for word embeddings, which captures the meaning of words by placing them in a continuous vector space.
 In this exercise, you will implement Word2Vec using NumPy and complete the missing parts of the code.
We will represent each word as a one-hot vector, meaning each word in the vocabulary is mapped to a unique binary vector with only one active (1) position.

import necessary libraries

In [11]:
import numpy as np
from collections import defaultdict

In [12]:
settings = {
	'window_size': 7,
	'n': 10,
	'epochs': 50,
	'learning_rate': 0.01,
	'embed_size': 50
}

In [13]:
class word2vec:
    def __init__(self, settings):
        """
        Initialize the Word2Vec model with given hyperparameters.
        """

        self.batch_size = settings['n']
        self.lr = settings['learning_rate']
        self.epochs = settings['epochs']
        self.window_size = settings['window_size']
        self.embed_size = settings['embed_size']
        self.words_list = None
        self.word_index = None
        self.index_word = None
        self.v_count = None
        self.w1 = None
        self.w2 = None

    def get_vocab(self, corpus):
        
        word_counts = defaultdict(int)

        for word in corpus:
            word_counts[word] += 1

        vocab_size = len(word_counts)
        words_list = list(word_counts.keys())
        
        word2idx = {word: i for i, word in enumerate(words_list)}
        idx2word = {i: word for i, word in enumerate(words_list)}
        
        return words_list, vocab_size, word2idx, idx2word
    
    # Function to get previous words in the window
    def prev_words(self, i, doc):
        out = []
        window = self.window_size//2
        for index in range(i - window, i):
            if index >= 0:
                out.append(self.word_index[doc[index]])
            else:
                out.append(0)  # 0 = <PAD>
        return out

    # Function to get next words in the window
    def next_words(self, i, doc):
        out = []
        window = self.window_size//2
        for index in range(i + 1, i + window + 1, 1):
            if index < len(doc):
                out.append(self.word_index[doc[index]])
            else:
                out.append(0)  # 0 = <PAD>
        return out
    
    def generate_training_data(self, corpus):
        
        # part 1
        self.words_list, self.v_count, self.word_index, self.index_word = self.get_vocab(corpus)
        print(self.words_list)
        print(self.v_count)
        print(self.word_index)
        print(self.index_word)
        
        # part 2
        X = []
        y = []
        for index, word in enumerate(corpus):
            prev = self.prev_words(index, corpus)
            next = self.next_words(index, corpus)
            assert len(prev) == len(next)
            X.append(prev + next)
            y.append([self.word_index[word]])
        
        X = np.array(X)
        y = np.array(y).flatten()

        training_data = list(zip(X, y))
        
        return training_data

    def word2onehot(self, word):
        
        word_vec = np.zeros(self.v_count)
        word_vec[self.word_index[word]] = 1
        return word_vec
    
    def train(self, training_data):

        # Initialize weights randomly
        self.w1 = np.random.rand(self.v_count, self.embed_size)
        self.w2 = np.random.rand(self.embed_size, self.v_count)

        for epoch in range(self.epochs):
            total_loss = 0

            # Initialize gradients and batch counter
            batch_grad_w1 = np.zeros_like(self.w1)
            batch_grad_w2 = np.zeros_like(self.w2)
            batch_count = 0

            for i, (x, target) in enumerate(training_data):

                # Generate one-hot encoded vectors for context words (x)
                context_vectors = [self.word2onehot(self.index_word[word_index]) for word_index in x]

                # Compute the average of context vectors to form the input vector (CBOW style)
                input_vector = np.mean(context_vectors, axis=0)

                # Generate one-hot encoded vector for the target word
                y_true = self.word2onehot(self.index_word[target])

                # Forward pass
                y_predicted, hidden_layer_output, _ = self.forward_pass(input_vector)

                # Call the backpropagation function for training
                loss, batch_grad_w1, batch_grad_w2, batch_count = self.backpropagation(
                    input_vector, y_true, hidden_layer_output, y_predicted,
                    batch_grad_w1, batch_grad_w2, batch_count,
                    training_complete=(i + 1 == len(training_data))
                )
                
                # Accumulate total loss
                total_loss += loss

            # Average loss over the entire training dataset
            total_loss /= len(training_data)
            print(f'Epoch {epoch+1}/{self.epochs} - Loss: {total_loss}')

    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    def forward_pass(self, input_vector):
        hidden_layer_output = np.dot(self.w1.T, input_vector)
        output_scores = np.dot(self.w2.T, hidden_layer_output)
        predicted_probabilities = self.softmax(output_scores)
        return predicted_probabilities, hidden_layer_output, output_scores
    
    def backpropagation(self, input_vector, y_true, hidden_layer_output, y_predicted, 
                         batch_grad_w1, batch_grad_w2, batch_count, training_complete=False):
    
        # Step 1: Calculate the error (difference between predicted and true labels)
        error = y_predicted - y_true
        loss = -np.log(np.dot(y_true, y_predicted) + 1e-9)
        
        # Step 2: Calculate gradients using backpropagation
        gradient_w2 = np.outer(hidden_layer_output, error)
        gradient_w1 = np.outer(input_vector, np.dot(self.w2, error))
        
        # Step 3: Accumulate gradients for Batch Processing
        batch_grad_w1 += gradient_w1
        batch_grad_w2 += gradient_w2
        batch_count += 1
        
        # Step 4: Update weights if batch is completed or training is finished
        if training_complete or (batch_count == self.batch_size):
            # Average the gradients for batch processing
            batch_grad_w1 /= batch_count
            batch_grad_w2 /= batch_count

            # Update weights using Gradient Descent
            self.w1 -= self.lr * batch_grad_w1
            self.w2 -= self.lr * batch_grad_w2

            # Reset gradients and batch count for the next batch
            batch_grad_w1 = np.zeros_like(self.w1)
            batch_grad_w2 = np.zeros_like(self.w2)
            batch_count = 0
        
        return loss, batch_grad_w1, batch_grad_w2, batch_count

    def word_vec(self, word):
        word_index = self.word_index[word]
        word_embedding = self.w1[word_index]
        return word_embedding 

    def vec_sim(self, word, top_n):
        v_w1 = self.word_vec(word)
        similarity = {}

        for other_word in self.words_list:
            if other_word == word:
                continue
            v_w2 = self.word_vec(other_word)
            sim = np.dot(v_w1, v_w2) / (np.linalg.norm(v_w1) * np.linalg.norm(v_w2))
            similarity[other_word] = sim

        words_sorted = sorted(similarity.items(), key=lambda x: x[1], reverse=True)

        for word, sim in words_sorted[:top_n]:
            print(f'{word}: {sim}')


In [14]:
text = "Natural language processing and machine learning open up fascinating possibilities, allowing machines to analyze,\
 understand, and respond to human language in ways that were once thought impossible."

In [15]:
corpus = [word.lower() for word in text.split()]
print(len(corpus))
print(corpus)

27
['natural', 'language', 'processing', 'and', 'machine', 'learning', 'open', 'up', 'fascinating', 'possibilities,', 'allowing', 'machines', 'to', 'analyze,', 'understand,', 'and', 'respond', 'to', 'human', 'language', 'in', 'ways', 'that', 'were', 'once', 'thought', 'impossible.']


In [16]:
w2v = word2vec(settings)

training_data = w2v.generate_training_data(corpus)

['natural', 'language', 'processing', 'and', 'machine', 'learning', 'open', 'up', 'fascinating', 'possibilities,', 'allowing', 'machines', 'to', 'analyze,', 'understand,', 'respond', 'human', 'in', 'ways', 'that', 'were', 'once', 'thought', 'impossible.']
24
{'natural': 0, 'language': 1, 'processing': 2, 'and': 3, 'machine': 4, 'learning': 5, 'open': 6, 'up': 7, 'fascinating': 8, 'possibilities,': 9, 'allowing': 10, 'machines': 11, 'to': 12, 'analyze,': 13, 'understand,': 14, 'respond': 15, 'human': 16, 'in': 17, 'ways': 18, 'that': 19, 'were': 20, 'once': 21, 'thought': 22, 'impossible.': 23}
{0: 'natural', 1: 'language', 2: 'processing', 3: 'and', 4: 'machine', 5: 'learning', 6: 'open', 7: 'up', 8: 'fascinating', 9: 'possibilities,', 10: 'allowing', 11: 'machines', 12: 'to', 13: 'analyze,', 14: 'understand,', 15: 'respond', 16: 'human', 17: 'in', 18: 'ways', 19: 'that', 20: 'were', 21: 'once', 22: 'thought', 23: 'impossible.'}


In [17]:
training_data[0]

(array([0, 0, 0, 1, 2, 3]), 0)

In [18]:
w2v.train(training_data)

Epoch 1/50 - Loss: 3.7688138857535627
Epoch 2/50 - Loss: 3.7475266431504934
Epoch 3/50 - Loss: 3.727425004757757
Epoch 4/50 - Loss: 3.7083857005902026
Epoch 5/50 - Loss: 3.690304408287265
Epoch 6/50 - Loss: 3.673092112660797
Epoch 7/50 - Loss: 3.656672264513844
Epoch 8/50 - Loss: 3.6409785464686917
Epoch 9/50 - Loss: 3.6259531030716365
Epoch 10/50 - Loss: 3.61154512853082
Epoch 11/50 - Loss: 3.5977097318750055
Epoch 12/50 - Loss: 3.584407018793008
Epoch 13/50 - Loss: 3.571601343847325
Epoch 14/50 - Loss: 3.559260697523864
Epoch 15/50 - Loss: 3.5473562006657358
Epoch 16/50 - Loss: 3.535861684950398
Epoch 17/50 - Loss: 3.5247533427178754
Epoch 18/50 - Loss: 3.51400943301594
Epoch 19/50 - Loss: 3.5036100334679228
Epoch 20/50 - Loss: 3.4935368296916702
Epoch 21/50 - Loss: 3.4837729356517086
Epoch 22/50 - Loss: 3.4743027396224244
Epoch 23/50 - Loss: 3.4651117714605113
Epoch 24/50 - Loss: 3.4561865876927955
Epoch 25/50 - Loss: 3.447514671568417
Epoch 26/50 - Loss: 3.439084345738219
Epoch 27/

In [19]:
word = "machine"
vec = w2v.word_vec(word)
print(word, vec)

# Find similar words
w2v.vec_sim("machine", 3)

machine [0.67068073 0.99543731 0.56070632 0.73354834 0.75107011 0.606797
 0.51764355 0.34729467 0.99476351 0.47573149 0.52345967 0.55659362
 0.49085817 0.2982996  0.17513302 0.68559302 0.4704974  0.64518061
 0.28583286 0.93896055 0.17000805 0.17098404 0.35764653 0.8952975
 0.52234538 0.56454427 0.33525377 0.41399879 0.84473658 0.81858914
 0.43517437 0.1615288  0.14951888 0.00437778 0.55518307 0.06282202
 0.77324133 0.13205872 0.16271532 0.5194837  0.04542994 0.57080346
 0.45820706 0.53248434 0.13738805 0.83413476 0.08023187 0.12455246
 0.70990604 0.73666295]
once: 0.8055594582915248
analyze,: 0.7959667728625622
possibilities,: 0.7953649478519569
